In [1]:
import numpy as np
import re

def parse_outcar_with_convergence(filename, n_atoms=47):
    # n_atoms=47
    positions = []
    forces = []
    energies = []
    convergence_flags = []
    
    # Read the OUTCAR file
    with open('OUTCAR', 'r') as f:
        lines = f.readlines()
    
    # Extract NELM (maximum number of electronic steps per ionic step)
    nelm = None
    for line in lines:
        if 'NELM' in line and '=' in line:
            try:
                nelm = int(line.strip().split('=')[1].split()[0])
                break
            except (IndexError, ValueError):
                continue
    if nelm is None:
        raise ValueError("NELM value not found in OUTCAR.")
    
    print(f"Detected NELM = {nelm}")
    
    # Iterate over lines to collect convergence info
    last_electronic_step = {}
    for i, line in enumerate(lines):
        match = re.search(r'Iteration\s+(\d+)\(\s*(\d+)\)', line)
        if match:
            ionic_step = int(match.group(1))
            electronic_step = int(match.group(2))
            last_electronic_step[ionic_step] = electronic_step
    
    # Mark convergence based on last electronic step in each ionic step
    for step in range(1, max(last_electronic_step.keys()) + 1):
        final_electronic = last_electronic_step.get(step, nelm)
        converged = int(final_electronic < nelm)
        convergence_flags.append(converged)
    
    # Extract positions, forces, and energies
    i = 0
    current_step = 0
    while i < len(lines):
        line = lines[i]
    
        if 'POSITION' in line and 'TOTAL-FORCE' in line:
            i += 2  # Skip dashed line
            pos_block = []
            frc_block = []
    
            for _ in range(n_atoms):
                parts = lines[i].strip().split()
                pos = list(map(float, parts[:3]))
                frc = list(map(float, parts[3:6]))
                pos_block.append(pos)
                frc_block.append(frc)
                i += 1
    
            # Search for energy
            energy = None
            for j in range(i, min(i + 20, len(lines))):
                if 'energy(sigma->0)' in lines[j]:
                    try:
                        energy = float(lines[j].strip().split('=')[-1])
                        break
                    except ValueError:
                        continue
    
            if energy is not None:
                positions.append(pos_block)
                forces.append(frc_block)
                energies.append(energy)
                current_step += 1
        else:
            i += 1
    
    return {
        'positions': np.array(positions),
        'forces': np.array(forces),
        'energies': np.array(energies),
        'convergence': np.array(convergence_flags[:len(positions)])
    }

# def save_converged_to_txt(data, filename='vasp_trajectory_converged.txt'):
#     positions = data['positions']
#     forces = data['forces']
#     energies = data['energies']
#     convergence = data['convergence']

#     n_steps = positions.shape[0]
#     n_atoms = positions.shape[1]

#     with open(filename, 'w') as f:
#         count = 0
#         for step in range(n_steps):
#             if convergence[step] == 1:
#                 f.write(f'  point= {count:6d}  {energies[step]:12.5f}\n')
#                 for atom in range(n_atoms):
#                     pos = positions[step][atom]
#                     frc = forces[step][atom]
#                     f.write(f'  {pos[0]:10.5f} {pos[1]:10.5f} {pos[2]:10.5f}  '
#                             f'{frc[0]:10.6f} {frc[1]:10.6f} {frc[2]:10.6f}\n')
#                 count += 1
#     print(f"Saved {count} converged steps to {filename}")

# def save_all(data, filename='all.txt'):
#     positions = data['positions']
#     forces = data['forces']
#     energies = data['energies']
#     with open(filename, 'w') as f:
#         for step in range(n_steps):
#             if convergence[step] == 1:
#                 f.write(f'  point= {count:6d}  {energies[step]:12.5f}\n')

                
# def main():
#     outcar_file = 'OUTCAR'
#     data = parse_outcar_with_convergence(outcar_file)
#     save_converged_to_txt(data)
#     # save_all(data)

# if __name__ == '__main__':
#     main()

In [3]:
data = parse_outcar_with_convergence('OUTCAR')
data['convergence']
idx = []
for i in range(len(data['convergence'])):
    if data['convergence'][i] == 0:
        idx.append(i)
print(idx)

Detected NELM = 200
[2, 3, 4, 5, 6, 7, 8, 9, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 468]
